# Cionic Metrics Analysis

This notebook performs gait metrics extraction and visualization from several data gait collections. The analysis includes:

- **Metrics Extraction**: Compute statistical measures (mean, std, CV) from sensor streams
- **Data Visualization**: Generate violin plots and statistical summary charts
- **Group Comparison**: Compare metrics across different experimental groups
- **Export Results**: Save metrics data to CSV for further analysis

### Input Requirements
- **Metadata List**: Defines experimental groups and their associated recordings
- **Stream Definitions**: Specifies which sensor streams and components to analyze
- **Authentication Token**: Provides access to the Cionic API

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cionic.gait_metrics import (
    STREAM_DEFINITIONS_FOR_STANDARD_METRICS,
    MetricsExtractor,
)
from cionic.plotting import (
    PLOTTING_METRIC_SPECIFICATION_LIST,
    TOP_RELEVANT_PLOTTING_METRICS,
    GroupedMetricsPlotter,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: separate by side

## Configuration Parameters

The following parameters define the analysis configuration. When executed via papermill (automated execution), these values are automatically overwritten with user-specified parameters from the metadata creation interface.

The `metadata_list` defines the experimental groups for comparison. Each group contains:
- **group_name**: Display name for the experimental condition
- **org_shortname**: Organization identifier (typically "cionic")
- **group_color**: Color for visualization (e.g., "steelblue", "sandybrown")
- **recordings**: List of specific data collections to include

Each recording specifies:
- **study_shortname**: Name of the study protocol
- **collection_num**: Unique identifier for the data collection session
- **label**: Specific activity or condition within the collection

In [ ]:
#######################################################################################
#
# Input values:
#     - tokenpath (overwritten by papermill)
#     - metadata_list (overwritten by papermill)
#     - stream_definitions
#
#######################################################################################

tokenpath = "/home/jovyan/cionic-data/token.json"

metadata_list = [
    {  # Can create an arbitrary number of groups
        "group_name": "Unstimulated",
        "org_shortname": "cionic",
        "group_color": "steelblue",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 55,
                "label": "unstimulated_walk",
            },
            {
                "study_shortname": "Parkinsons",
                "collection_num": 59,
                "label": "unstimulated_walk",
            },
        ],
    },
    {
        "group_name": "Stimulated",
        "org_shortname": "cionic",
        "group_color": "sandybrown",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 59,
                "label": "stim_walk",
            },
            {
                "study_shortname": "Parkinsons",
                "collection_num": 86,
                "label": "stim_walk",
            },
        ],
    },
]

## Stream Definitions Configuration

Stream definitions specify which sensor data streams to analyze for metrics extraction. Each definition includes:
- **stream**: Type of sensor data (e.g., 'euler', 'gyro', 'accel')
- **position**: Anatomical location (e.g., 'thigh', 'shank', 'knee_joint')
- **component**: Specific axis or measurement (e.g., 'x', 'y', 'z', 'knee_flexion')

Additional stream definitions can be added by appending dictionaries of the form:
```python
{'stream': 'euler', 'position': 'thigh', 'component': 'x'}
```

In [ ]:
stream_definitions = STREAM_DEFINITIONS_FOR_STANDARD_METRICS
pprint(stream_definitions)

## Metrics Extraction

Create the `MetricsExtractor` instance and compute gait metrics from the configured data streams. This process:

1. **Loads Data**: Retrieves sensor data for each specified recording
2. **Processes Streams**: Applies signal processing to extract meaningful metrics
3. **Aggregates Results**: Combines metrics across all groups and recordings into a single DataFrame

In [ ]:
metric_extractor = MetricsExtractor(
    metadata_list=metadata_list,
    stream_definitions=stream_definitions,
    tokenpath=tokenpath,
)
all_metrics = metric_extractor.extract_metrics()

## Data Export

Save the extracted metrics to a CSV file for further analysis and provide a summary of the results:

In [ ]:
# Create filename with timestamp
csv_filename = "metrics.csv"

# Save to CSV
all_metrics.to_csv(csv_filename, index=False)
print(f"Metrics saved to: {csv_filename}")
print(f"Total rows: {len(all_metrics)}")
print(f"Total columns: {len(all_metrics.columns)}")

display(all_metrics.head())

## Visualization Configuration

Configure which metrics to visualize using predefined plotting specifications. Each metric specification includes:

- **title**: Descriptive figure title for display
- **y_label**: Units and measurement description for the y-axis
- **metric_column**: Column name from the metrics DataFrame (e.g., 'mean_value', 'std_value', 'cv_value')
- **position**: Anatomical position being analyzed
- **component**: Specific measurement component or axis

New metrics can be added to the visualization list by appending a dictionary of the form:
```python
new_metric = {
    "title": "Thigh Mean (Sagittal)",  # Descriptive figure title
    "y_label": "Euler, (degrees)",  # Figure y label
    "metric_column": "mean_value",  # Valid metric column in all_metrics
    "position": "thigh",  # Valid position, e.g., thigh, knee_joint, etc.
    "component": "x",  # Valid component, e.g., x, y, knee_flexion
}
```

In [ ]:
metric_specification_list = PLOTTING_METRIC_SPECIFICATION_LIST

# Comment this out to plot broader metrics list
metric_specification_list = [
    m for m in metric_specification_list if m["title"] in TOP_RELEVANT_PLOTTING_METRICS
]

print("Plotting metrics: ")
for metric_specification in metric_specification_list:
    print(f"- {metric_specification['title']}")

## Violin Plot Visualization

Generate violin plots to show the distribution of metrics across experimental groups. Violin plots display:

- **Distribution Shape**: Full probability density of the data
- **Quartiles**: Box plot elements showing median and quartile boundaries  
- **Group Comparisons**: Side-by-side comparison of different experimental conditions
- **Individual Data Points**: Optional overlay of actual measurements

These plots are ideal for comparing variability and central tendencies between groups.

In [ ]:
plotter = GroupedMetricsPlotter(metrics=all_metrics)
for metric_specification in metric_specification_list:
    fig, ax = plotter.violin_plot(metric_specification=metric_specification)
    if fig is not None and ax is not None:
        # Additional plotting customization can be added here.
        plt.tight_layout()
        plt.show()

## Statistical Summary Bar Charts

Create bar charts showing statistical summaries for each metric and group. Available statistics include:

- **"mean"**: Average value across all measurements in each group
- **"std"**: Standard deviation showing absolute variability
- **"cv"**: Coefficient of variation (std/mean) showing relative variability as a percentage

These charts provide a quantitative comparison of central tendencies and variability between experimental groups.

In [ ]:
# Options: "mean", "std", "cv"
statistic = "cv"

for metric_specification in metric_specification_list:
    fig, ax = plotter.statistical_summary_bar_plot(
        metric_specification=metric_specification, statistic=statistic
    )
    if fig is not None and ax is not None:
        # Additional plotting customization can be added here.
        plt.tight_layout()
        plt.show()